# BACE-1 pIC50：活性打分 → 分子生成 → 分子优化

由溶解度 `07` / 旧版 `08_esol_generate_optimize` **换成活性任务**。流水线骨架不变，换的是 **打什么分、优化什么数**。

| 角色 | 本 notebook 里是谁 | 易混点 |
|------|---------------------|--------|
| **主靶** | BACE-1（β-分泌酶） | 只预测「对这个蛋白好不好」，不是 ChEMBL 里所有活性 |
| **主读出** | **pIC50 回归**（越大越强） | 不是浓度 IC50 本身；也先不做 0/1 分类 |
| **打分器** | SMILES **Transformer 回归**（对齐 07 里最准的那类模型） | 不是 RF；也不是下面写 SMILES 的 GPT |
| **生成器** | 因果 Transformer（MolGPT），在 **无 pIC50 的大 SMILES 库** 上预训练 | 学的是「怎么写合法分子」，不是活性 |
| **优化器** | Encoder–Decoder：弱活性种子 → 更高 pIC50 的近邻 | **必须有起点分子**；生成则没有起点 |

**两套数据（不要当成从 ZINC 里按有无标签切开）：**

- 带实验 pIC50：MoleculeNet **BACE**（约 1513 条）→ 只训打分器 + 构造优化对
- 无活性标签：本地 ZINC **约 25 万全集** SMILES（不再抽样）→ 只训生成器（无 pIC50）

打分器给新分子的数是 **模型分**，不是实验。优化若成功，先说「打分器认为变强了」。

In [ ]:
import os
import re
import math
import csv
import random
import urllib.request
from pathlib import Path
from collections import Counter, defaultdict   # defaultdict，dict子类

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.metrics import mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

RDLogger.DisableLog("rdApp.*")
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

_NB_DIR = Path.cwd()   # 当前工作目录


def _find_data_dir():
    for candidate in (_NB_DIR / "data", _NB_DIR / ".." / "data", Path(r"d:\MachineLearning\aidd\data")):
        candidate = candidate.resolve()    # 把 .. 收成真实绝对路径
        if candidate.exists():
            return candidate
    data_dir = (_NB_DIR / ".." / "data").resolve()
    data_dir.mkdir(parents=True, exist_ok=True)
    return data_dir


DATA_DIR = _find_data_dir()
BACE_PATH = DATA_DIR / "bace_pic50.csv"
ZINC_RAW = DATA_DIR / "zinc250k_raw.csv"      # 本地 ZINC 全集（约 25 万）
ZINC_PATH = DATA_DIR / "zinc_gen_full.csv"    # 全量 canonical 缓存（不再抽子集）
ZINC_URL = (
    "https://raw.githubusercontent.com/aspuru-guzik-group/chemical_vae/"
    "master/models/zinc/250k_rndm_zinc_drugs_clean_3.csv"
)

# 主靶 / 主读出写进变量名，避免和 07 的 logS 混用
TARGET_NAME = "BACE-1"
READOUT = "pIC50"

RANDOM_STATE = 42
MAX_LEN = 128          # BACE 抑制剂偏长，比 ESOL 的 100 更宽；超长仍会截断
EPOCHS_SCORER = 20     # 打分器（回归 MSE）；可比 GPT 多训几轮，数据只有约 1.2k
EPOCHS_GPT = 20         # 全量 ZINC（约 25 万）× 20 epoch；CPU 很慢，冒烟可改 EPOCHS_GPT=2
EPOCHS_COND = 15       # 仅在 BACE 上微调条件符；最低/最高约 30%，中间丢掉
EPOCHS_SEQ2SEQ = 15
# 生成语料：默认用 ZINC 全量；冒烟测试可设 GEN_MAX_N=2000
GEN_MAX_N = None         # None=全量；整数=最多保留这么多条（仅调试）
NEW_SAMPLE = 50          # 生成采样条数（评有效率 / 新颖性）
DEVICE = torch.device("cpu")
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

print("cwd:", os.getcwd())
print("data dir:", DATA_DIR)
print("target:", TARGET_NAME, "readout:", READOUT)
print("torch:", torch.__version__, "device:", DEVICE)

## 1. 下载 / 读取两套数据 + scaffold 划分

BACE 表里同时有 `pIC50` 和 `Class`。本子 **训练只用 pIC50**；`Class` 只打印对照，避免「回归、分类一起训」却只说做了回归。

划分：**Murcko 骨架 split**，不是 `train_test_split` 随机切。活性数据里同骨架类似物很多，随机划分会把近亲漏到测试集，RMSE/R² 虚高。性质预测（ESOL）也有这个问题，活性上更严重。


In [ ]:
def download(url, dest: Path, timeout=90):
    dest.parent.mkdir(parents=True, exist_ok=True)   # 保存数据路径的父文件夹存
    print("downloading", url)
    urllib.request.urlretrieve(url, dest)            # 标准库把该 URL 的内容写到 dest 这个文件
    print("saved", dest, dest.stat().st_size, "bytes")   


def ensure_bace(path: Path) -> Path:
    if path.exists():
        return path
    # MoleculeNet 原表列很多；我们只落 SMILES / Class / pIC50
    tmp = path.with_name("bace_full_tmp.csv")      # 算出临时文件路径，和目标path在同一目录，只换文件名为bace_full_tmp.csv
    download("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/bace.csv", tmp)
    raw = pd.read_csv(tmp, usecols=["mol", "CID", "Class", "pIC50"])
    raw.to_csv(path, index=False)
    tmp.unlink(missing_ok=True)    # 删掉临时全量表
    return path

# 不同写法smiles转为相同mol分子结构，最终生成标准smiles
def canonical_smiles(smi):
    mol = Chem.MolFromSmiles(smi)    # 写错、价态不合法 → 得不到 mol → 返回 None，后面丢掉
    # canonical=True表示同一分子尽量用同一种字符串（原子顺序、环编号会规范），生成标准smiles判断不同写法两个smiles是否是同一分子
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None  


# 按 Murcko 骨架分组，同一骨架整组进 train 或 test，避免随机划分把近亲漏到测试集。
def scaffold_split(smiles_list, test_size=0.2, seed=42):
    buckets = defaultdict(list)      # 创建字典，defaultdict(list)表示键不存在时，值自动放一个空列表,方便直接append添加元素
    for idx, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)            # 先解析成分子图，才能剥骨架 
        if mol is None:
            key = f"invalid_{idx}"
        else:
            try:
                key = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)   # 剥掉侧链，只留核心环系统，得到骨架 SMILES。相同骨架的分子拿到同一个key
            except Exception:
                key = f"none_{idx}"             # 个别结构 Murcko 会失败：同样单独成组，不让整段划分崩掉。
        buckets[key].append(idx)
    rng = np.random.RandomState(seed)
    keys = list(buckets.keys())          # buckets.keys() 是这个字典里所有键
    rng.shuffle(keys)
    num_test = max(1, int(len(smiles_list) * test_size))
    idx_test, idx_train = [], []
    for scaffold in keys:
        if len(idx_test) < num_test:
            idx_test.extend(buckets[scaffold])    # extend(iterable)：把可迭代对象拆开，逐个元素接上去（和 += 一组元素类似）
        else:
            idx_train.extend(buckets[scaffold])
    return np.array(idx_train), np.array(idx_test)


# 保存mol结构合法、pic50存在的 smiles、lable、class
ensure_bace(BACE_PATH)
bace = pd.read_csv(BACE_PATH)
# 只要主读出；Class 留下做描述，不进损失函数
keep_smi, keep_y, keep_cls = [], [], []
for smi, pic50, cls in zip(bace["mol"], bace["pIC50"], bace["Class"]):
    can = canonical_smiles(str(smi))
    if can is None or pd.isna(pic50):
        continue
    keep_smi.append(can)
    keep_y.append(float(pic50))
    keep_cls.append(int(cls))

y = np.asarray(keep_y, dtype=np.float32)
cls_arr = np.asarray(keep_cls, dtype=np.int64)
idx_train, idx_test = scaffold_split(keep_smi, test_size=0.2, seed=RANDOM_STATE)  

smiles_arr = np.asarray(keep_smi, dtype=object)
smiles_bace_train = smiles_arr[idx_train].tolist()
smiles_bace_test = smiles_arr[idx_test].tolist()
y_train, y_test = y[idx_train], y[idx_test]
print(f"BACE kept {len(keep_smi)} | train {len(smiles_bace_train)} | test {len(smiles_bace_test)}")
print(f"{READOUT} train median={np.median(y_train):.3f}  min={y_train.min():.2f}  max={y_train.max():.2f}")
print("说明：Class≈「是否算抑制剂」；pIC50≈「有多强」。")

## 1b. 无标签生成语料（ZINC 约 25 万 **全量**）

生成器 **没有 pIC50**，只学合法、偏药像的写法。  
顺序：已有 `zinc_gen_full.csv`（全量 canonical 缓存）→ 否则从本地 `zinc250k_raw.csv`  canonical 全部并缓存 → 再没有才联网下载全集 → 仍失败才回退 BACE-train SMILES。

全量约 25 万条，CPU 上训 MolGPT 会比原先 10 万子集更慢。冒烟测试把 `GEN_MAX_N` 设成小整数即可。


In [ ]:
# 先准备全量合法 SMILES，再 load（供缓存 / 调试截断）
def canonicalize_smiles_series(series, max_n=None, seed=RANDOM_STATE):
    """对 series 做 canonical；max_n=None 表示全量，否则打乱后最多保留 max_n 条。"""
    rng = np.random.RandomState(seed)
    pool = series.astype(str).str.replace("\n", "", regex=False).str.strip().tolist()
    if max_n is not None:
        rng.shuffle(pool)
    smiles_list = []
    for smi in pool:
        can = canonical_smiles(smi)
        if can:
            smiles_list.append(can)
        if max_n is not None and len(smiles_list) >= max_n:
            break
    return smiles_list


# 加载 ZINC：默认全量（约 25 万），不再随机抽 10 万 / 1 万子集
def load_zinc(max_n=GEN_MAX_N):
    # 1) 已有全量缓存
    if ZINC_PATH.exists():
        smiles_list = pd.read_csv(ZINC_PATH)["smiles"].astype(str).tolist()
        if max_n is None or len(smiles_list) >= max_n:
            out = smiles_list if max_n is None else smiles_list[:max_n]
            print("use", ZINC_PATH, "have", len(smiles_list), "take", len(out))
            return out
        print("cache too small", len(smiles_list), "<", max_n, "-> rebuild from full set")

    # 2) 本地 raw 全集
    if ZINC_RAW.exists():
        print("loading FULL set from", ZINC_RAW, "max_n=", max_n)
        df = pd.read_csv(ZINC_RAW)
        smiles_col = "smiles" if "smiles" in df.columns else df.columns[0]
        smiles_list = canonicalize_smiles_series(df[smiles_col], max_n=None, seed=RANDOM_STATE)
        pd.DataFrame({"smiles": smiles_list}).to_csv(ZINC_PATH, index=False)
        print("wrote", ZINC_PATH, "n=", len(smiles_list))
        if max_n is not None:
            return smiles_list[:max_n]
        return smiles_list

    # 3) 下载全集
    try:
        print("download full ZINC ->", ZINC_RAW)
        download(ZINC_URL, ZINC_RAW, timeout=180)
        df = pd.read_csv(ZINC_RAW)
        smiles_col = "smiles" if "smiles" in df.columns else df.columns[0]
        smiles_list = canonicalize_smiles_series(df[smiles_col], max_n=None, seed=RANDOM_STATE)
        pd.DataFrame({"smiles": smiles_list}).to_csv(ZINC_PATH, index=False)
        print("wrote", ZINC_PATH, "n=", len(smiles_list))
        if max_n is not None:
            return smiles_list[:max_n]
        return smiles_list
    except Exception as e:
        print("ZINC download failed:", type(e).__name__, e)

    print("FALLBACK: BACE train SMILES only")
    return list(smiles_bace_train)


smiles_gen = load_zinc()
print("gen data_num:", len(smiles_gen), "example:", smiles_gen[0][:80])


## 2. 两套 Tokenizer（高发混淆点）

- `tok_score`：只在 **BACE 训练集** 上建词表，给打分器编码。  
- `tok_gen`：在 **生成语料 ∪ BACE 训练 SMILES** 上建词表，给 GPT / Seq2Seq。  

生成出来的分子去打分时，必须走 `tok_score.encode`。ZINC 上见过、BACE 词表没有的 token 会变成 `<unk>`，打分会偏——这是分布偏移，不是 bug。


In [ ]:
class SMILESTokenizer:
    def __init__(self, max_len=128, extra_special=None):
        self.max_len = max_len
        self.pattern = (
            r"(\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|"
            r"\(|\)|\.|=|#|-|\+|\\\\|/|:|~|@|\?|>|\*|\\*|\$|%[0-9]{2}|[0-9])"
        )
        self.regex = re.compile(self.pattern)
        self.pad_token, self.unk_token = "<pad>", "<unk>"
        self.sos_token, self.eos_token = "<sos>", "<eos>"
        self.special_tokens = [self.pad_token, self.unk_token, self.sos_token, self.eos_token]
        if extra_special:
            self.special_tokens = self.special_tokens + list(extra_special)
        self.vocab = {token: idx for idx, token in enumerate(self.special_tokens)}
        self.inv_vocab = {idx: token for token, idx in self.vocab.items()}

    # 分析器：将smiles → token
    def tokenize(self, smiles: str):
        return self.regex.findall(smiles)     # Pattern.findall寻找所有匹配，按顺序放进  列表

    # 构建 token——ID对应的两张词表，供encode和decode对照翻译
    def build_vocab(self, smiles_list):
        counter = Counter()        # Counter({'C': 2, 'O': 1}) 字典的子类，专门做元素出现次数统计，键是token,值是次数
        for smi in smiles_list:
            counter.update(self.tokenize(smi))   # update(可迭代对象)，对列表里每个元素 +1（不是替换整个字典）
        for token, _ in counter.most_common():  # counter.most_common() 按次数从高到低返回(token, 次数)二元组，让常见符号拿到较小的 ID
            if token not in self.vocab:
                idx = len(self.vocab)
                self.vocab[token] = idx
                self.inv_vocab[idx] = token
        return self

    # 将smiles 分词为 token，在查阅词表将token 转换为 ids，加入特殊字段 ids，根据pad=True/False决定是否padding补ids维度，最终返回 ids列表
    def encode(self, smiles: str, add_special=True, pad=True):
        tokens = self.tokenize(smiles)
        if len(tokens) > self.max_len - 2:
            tokens = tokens[: self.max_len - 2]
        unk_id = self.vocab[self.unk_token]
        ids = [self.vocab.get(token, unk_id) for token in tokens]
        if add_special:
            ids = [self.vocab[self.sos_token]] + ids + [self.vocab[self.eos_token]]
        if pad and len(ids) < self.max_len:
            ids = ids + [self.vocab[self.pad_token]] * (self.max_len - len(ids))
        return ids

    # 查阅词表 将ids → token，跳过特殊token → 最后连成一条smiles
    def decode(self, ids, skip_special=True):
        tokens_out = []
        for token_id in ids:
            token = self.inv_vocab.get(int(token_id), self.unk_token)
            # <act_high> 这类控制符也要丢掉，否则续写结果开头粘着标签
            if skip_special and (token in self.special_tokens or str(token).startswith("<")):
                continue
            tokens_out.append(token)
        return "".join(tokens_out)



ACT_TAGS = ("<act_high>", "<act_low>")   # 条件生成会用到；先放进生成词表，避免微调时膨胀 Embedding 对不上旧权重
tok_score = SMILESTokenizer(max_len=MAX_LEN).build_vocab(smiles_bace_train)   # 链式写法，左侧先实例化一个分析器对象，直接调用其构建词表方法，最终返回同一台分析器，词表已建好
tok_gen = SMILESTokenizer(max_len=MAX_LEN, extra_special=list(ACT_TAGS)).build_vocab(
    list(smiles_gen) + list(smiles_bace_train)
)   # BACE和ZINC数据集共同拿来构造生成器的词表
PAD_ID_S = tok_score.vocab[tok_score.pad_token]
PAD_ID_G = tok_gen.vocab[tok_gen.pad_token]
SOS_ID_G = tok_gen.vocab[tok_gen.sos_token]  # 两层取值，从内往外看。内层tok_gen.sos_token这个对象上存的开始符字符串self.sos_token="<sos>"
EOS_ID_G = tok_gen.vocab[tok_gen.eos_token]  # 外层tok_gen.vocab这个对象的字典用键"<sos>"取对应的id值
HIGH_ID = tok_gen.vocab["<act_high>"]
LOW_ID = tok_gen.vocab["<act_low>"]
print("score vocab", len(tok_score.vocab), "pad", PAD_ID_S)
print("gen vocab  ", len(tok_gen.vocab), "pad", PAD_ID_G, "act_high", HIGH_ID)

## 3. 打分器：双向 Transformer 回归 pIC50（不是 GPT）

与 07 的 `TransformerRegressor` 同类：**整句双向注意** → 平均池化 → 一个实数。  
损失是 **MSE**，指标 RMSE / R²。  

后面的 MolGPT 则是 **因果 mask + 交叉熵**。两个模型都叫 Transformer，任务完全不同：一个读分子打分，一个写分子。


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, hidden):
        return hidden + self.pe[:, : hidden.size(1), :]


class TransformerRegressor(nn.Module):
    """双向 Encoder + 回归头 → pIC50。"""

    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_feedforward=256, pad_idx=0, dropout=0.1):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, input_ids):   # input_ids形状[B,L]
        pad_mask = input_ids.eq(self.pad_idx)   # .eq() 是PyTorch张量上的方法，逐元素相等比较，返回一个input_ids同形状的布尔张量
        hidden = self.embedding(input_ids) * math.sqrt(self.d_model)   # Embedding 后乘 sqrt(d_model)：原论文缩放，避免与位置编码量级差太大
        hidden = self.pos_encoder(hidden)
        hidden = self.encoder(hidden, src_key_padding_mask=pad_mask)   # 传进encoder计算，src_key_padding_mask告诉注意力：哪些位置是 pad，不要当「内容」来看，输出形状仍是[B, L, D]
        
        # 【关键】只对非 pad 位置做平均池化，得到「整条分子」的句向量  [B, L, D]→[B, D] ,相当于GNN中的readout
        # 句向量要同时满足「一个分子一个向量、向量里仍是特征（才能回归 pIC50）、长度固定（才能接同一套 head）」
        non_pad_mask = (~pad_mask).unsqueeze(-1).float()    #  [B, L]→[B, L, 1]
        # 每个分子的有效token向量之和 /每个分子有效token个数 =「每个有效位置贡献相等」的句向量
        # hidden * non_pad_mask：padding不是分子内容，求和前乘 0 清除
        feat = (hidden * non_pad_mask).sum(dim=1) / non_pad_mask.sum(dim=1).clamp(min=1e-9)    
        return self.head(feat).squeeze(-1)


class Pic50Dataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer):
        self.smiles_list = list(smiles_list)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        input_ids = self.tokenizer.encode(self.smiles_list[idx], add_special=True, pad=True)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }


train_loader_scorer = DataLoader(Pic50Dataset(smiles_bace_train, y_train, tok_score), batch_size=32, shuffle=True)
test_loader_scorer = DataLoader(Pic50Dataset(smiles_bace_test, y_test, tok_score), batch_size=32)

scorer = TransformerRegressor(len(tok_score.vocab), pad_idx=PAD_ID_S).to(DEVICE)
opt_scorer = torch.optim.Adam(scorer.parameters(), lr=1e-3)
mse = nn.MSELoss()
history_scorer = []


def eval_scorer(loader):
    scorer.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in loader:     # batch为包含"input_ids"和"labels"两键的字典，对应的值为一批smiles转换后的ids和labels堆叠而成，形状[B,L]和[B]
            input_ids = batch["input_ids"].to(DEVICE)
            pred = scorer(input_ids).cpu().numpy()
            y_true.extend(batch["labels"].numpy().tolist())
            y_pred.extend(pred.tolist())
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return rmse, r2, np.asarray(y_true), np.asarray(y_pred)


for epoch in range(1, EPOCHS_SCORER + 1):
    scorer.train()
    total, n_sample = 0.0, 0
    for batch in train_loader_scorer:
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # 标准步：清梯度 → 前向 → 损失 → 反向 →（裁剪）→ 更新
        opt_scorer.zero_grad()
        pred = scorer(input_ids)
        loss = mse(pred, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(scorer.parameters(), 1.0)   # 梯度裁剪：限制更新幅度，减轻 Transformer 训练中的梯度爆炸
        opt_scorer.step()

        total += loss.item() * len(labels)    # 按样本数加权，与 MLP 一致
        n_sample += len(labels)
    tr_mse = total / max(n_sample, 1)
    te_rmse, te_r2, y_true_e, y_pred_e = eval_scorer(test_loader_scorer)
    te_mse = float(mean_squared_error(y_true_e, y_pred_e))
    history_scorer.append((tr_mse, te_mse))
    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS_SCORER:
        print(f"[Scorer TF] epoch {epoch:3d}  train_MSE={tr_mse:.4f}  test_MSE={te_mse:.4f}")

sc_rmse, sc_r2, y_true_te, y_pred_te = eval_scorer(test_loader_scorer)
print(f"\nBACE {READOUT}  Transformer  scaffold-test  RMSE={sc_rmse:.4f}  R2={sc_r2:.4f}")


hist = np.array(history_scorer)
plt.figure(figsize=(5, 3))
plt.plot(hist[:, 0], label="train MSE")
plt.plot(hist[:, 1], label="test MSE")
plt.xlabel("epoch")
plt.ylabel("MSE (pIC50)")
plt.legend()
plt.title("BACE pIC50 scorer")
plt.tight_layout()
plt.show()


@torch.no_grad()
def score_pic50(smiles_list):   # 给任意 SMILES 估 pIC50
    scorer.eval()
    scores, valid_smiles = [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        can = Chem.MolToSmiles(mol, canonical=True)
        # tok_score.encode(can)返回已是列表，再套一层[]转为二维张量[B,L]，与模型forward要求形状相同，不过B维是1
        input_ids = torch.tensor([tok_score.encode(can)], dtype=torch.long, device=DEVICE)
        pred = float(scorer(input_ids).cpu().item())
        scores.append(pred)
        valid_smiles.append(can)
    return np.asarray(scores, dtype=np.float64), valid_smiles

## 4. 无条件生成（MolGPT）

自回归：每步只预测下一个 token，用因果 mask 禁止看后面。  
预训练数据是 **ZINC 风格 SMILES**，不要期望它写出已经像 BACE 抑制剂的肽模拟物；那是打分 + 筛选 / 优化的事。


In [ ]:
# Encoder + causal mask = GPT 式 decoder-only。不要和上面的双向回归打分器当成同一个模型
class MolGPT(nn.Module):

    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=3, max_len=128, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = nn.Embedding(max_len, d_model)   # 可学习位置编码；打分器是正弦 PositionalEncoding。同角色、不同实现
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256, batch_first=True, dropout=0.1,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)   # 输出[B, L, V]。 Linear默认只动最后一维，把encoder最后一维映射到vocab_size，得到每个位置下一个token是哪个

    # 【因果掩码矩阵】 生成一张上三角掩码矩阵：写第 t 个 token 时，不能偷看第 t+1 及以后的内容
    def _causal_mask(self, seq_len, device):   # seq_len:当前序列长度，本batch内最长那条的token数
        # diagonal=1表示主对角线及以下全变成 False，只保留主对角线上方的 True。
        # PyTorch 的 TransformerEncoder（当 attn_mask 传布尔值时）里 True = 屏蔽，所以上三角的 True 恰好挡住"未来"，让每个位置只能 attend 自己和左边。
        return torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1) 
    '''
    位置 0 1 2 3     注意力机制在计算时当前位置的信息来源
    0  [F T T T]   # 位置 0 看不见 1、2、3
    1  [F F T T]   # 位置 1 看不见 2、3
    2  [F F F T]   # 位置 2 看不见 3
    3  [F F F F]   # 位置 3 没有"未来"
    '''

    def forward(self, input_ids):   # input_ids形状[B, L]，
        batch_size, seq_len = input_ids.shape
        pad_mask = input_ids.eq(self.pad_idx)
        pos = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)   # 构造 位置索引 张量，形状 [B, L]
        hidden = self.embedding(input_ids) * math.sqrt(self.d_model) + self.pos_encoder(pos)   # 把 token id 转成向量，并加上位置信息
        # 因果掩码矩阵 和 padding掩码矩阵都在 encoder层内使用
        hidden = self.encoder(hidden, mask=self._causal_mask(seq_len, input_ids.device), src_key_padding_mask=pad_mask)
        return self.head(hidden)

    @torch.no_grad()
    def generate(self, max_new_tokens=100, temperature=1.0, prefix_ids=None):   # temperature：控制随机程度（越大越乱，越小越「挑概率最大的」）
        self.eval()
        if prefix_ids is None:
            input_ids = torch.tensor([[SOS_ID_G]], dtype=torch.long, device=DEVICE)  # [SOS_ID_G]：一条长度为1的序列，没有batch维。forward要的是二维[B,L]：一条batch、一条序列。外面那层 [] 就是补上 B=1
        else:
            input_ids = torch.tensor([prefix_ids], dtype=torch.long, device=DEVICE)  # prefix_ids 本身已经是一层列表（序列维），只需套一层补batch维
        for _ in range(max_new_tokens):
            # 这一步真正喂给forward的条件序列，形状[1, min(当前token长度, 128)]
            # .num_embeddings是nn.Embedding的属性：位置表有多少行/当前能看token数
            cond_ids = input_ids[:, -self.pos_encoder.num_embeddings :]  
            # logits：下一个 token 在词表上的未归一化分数（已按温度缩放），[1, V]。
            # self(cond_ids)：[1, l] → [1, l, V],
            # [:, -1, :]索引切片取最后一个token位置对应的V维向量（代表下个token）
            logits = self(cond_ids)[:, -1, :] / max(temperature, 1e-6)
            # 在最后一维词表维上做softmax，把V个数变为概率，和为1，从中抽一个类别
            next_id = torch.multinomial(F.softmax(logits, dim=-1), 1)
            input_ids = torch.cat([input_ids, next_id], dim=1)
            if int(next_id.item()) == EOS_ID_G:
                break
        return input_ids[0].tolist()


class GenDataset(Dataset):
    def __init__(self, smiles_list, tokenizer):
        self.smiles_list = smiles_list
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):
        ids = self.tokenizer.encode(self.smiles_list[idx], add_special=True, pad=False)  # 不padding补L维数量
        input_ids = torch.tensor(ids[:-1], dtype=torch.long)   # 左闭右开  去掉最后一位
        target_ids = torch.tensor(ids[1:], dtype=torch.long)   # 去掉第一位   训练目标！！！！！与输入ids向后错开一位，即预测下一个token的ids
        return input_ids, target_ids

# 是DataLoader的collate_fn：把一个batch里长度不同的 (input_ids,target_ids)在右边补PAD，再堆成两个二维张量，好送进模型dataloader循环。
def collate_shift(batch, pad_id=PAD_ID_G):
    batch_max_len = max(len(input_ids) for input_ids, _ in batch)  # target_ids与input_ids长度相同
    input_ids_list, target_ids_list = [], []
    for input_ids, target_ids in batch:
        n_pad = batch_max_len - len(input_ids)
        pad_ids = torch.full((n_pad,), pad_id, dtype=torch.long)  # 造一段长度为 n_pad、值全是 pad_id 的一维张量
        input_ids_list.append(torch.cat([input_ids, pad_ids]))   # 把 PAD 接在右边（序列末尾），再放进列表。
        target_ids_list.append(torch.cat([target_ids, pad_ids]))
    return torch.stack(input_ids_list), torch.stack(target_ids_list)


train_loader_gpt = DataLoader(
    GenDataset(smiles_gen, tok_gen), batch_size=32, shuffle=True, collate_fn=collate_shift,
)
gpt = MolGPT(len(tok_gen.vocab), max_len=MAX_LEN, pad_idx=PAD_ID_G).to(DEVICE)
opt_gpt = torch.optim.Adam(gpt.parameters(), lr=1e-3)
# ce(pred, target)交叉熵误差的输入pred为为归一化（softmax）的类别向量，要求形状[N,V],N=B x L（把 batch 和序列长度摊平后的数目）
ce = nn.CrossEntropyLoss(ignore_index=PAD_ID_G)   # 计算交叉熵误差时忽略标签维为PAD_ID_G的位置
history_gpt = []

for epoch in range(1, EPOCHS_GPT + 1):
    gpt.train()
    total = 0.0
    for input_ids, target_ids in train_loader_gpt:   # dataset返回是二元组，直接解包就行，无需像打分器中for batch字典，再通过键取值
        input_ids, target_ids = input_ids.to(DEVICE), target_ids.to(DEVICE)
        opt_gpt.zero_grad()
        # 传入模型batch[B, L]，输出每个位置长度为 V 的向量[B, L, V]
        logits = gpt(input_ids)    
        # 计算损失值
        loss = ce(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))
        # 反向传播计算梯度
        loss.backward()
        # 梯度裁剪。所有参数梯度拼起来的 L2 范数大于 1，就整体缩小到 1。减轻 Transformer 里梯度爆炸，更新不会一步迈太大。
        nn.utils.clip_grad_norm_(gpt.parameters(), 1.0) 
        opt_gpt.step()
        total += loss.item()
    train_ce = total / max(len(train_loader_gpt), 1)
    history_gpt.append(train_ce)
    if epoch == 1 or epoch % 2 == 0 or epoch == EPOCHS_GPT:
        print(f"[MolGPT] epoch {epoch:3d}  CE={train_ce:.4f}")

plt.figure(figsize=(5, 3))
plt.plot(history_gpt, color="#c45c26")
plt.xlabel("epoch")
plt.ylabel("token CE")
plt.title("Unconditional MolGPT (unlabeled SMILES)")
plt.tight_layout()
plt.show()

In [ ]:
train_gen_set = set(smiles_gen)
train_bace_set = set(smiles_bace_train)


def evaluate_generated(smiles_list, tag="uncond"):
    valid_smiles, novel_gen, novel_bace = [], 0, 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        can = Chem.MolToSmiles(mol, canonical=True)   # 标准smiles写法
        valid_smiles.append(can)
        novel_gen += int(can not in train_gen_set)
        novel_bace += int(can not in train_bace_set)
    print(f"[{tag}] 采样 {len(smiles_list)}  有效 {len(valid_smiles)}  "
          f"相对生成语料新颖 {novel_gen}/{len(valid_smiles) if valid_smiles else 0}  "
          f"相对 BACE-train 新颖 {novel_bace}/{len(valid_smiles) if valid_smiles else 0}")
    if not valid_smiles:
        return
    scores, _ = score_pic50(valid_smiles)
    print(f"  打分器预测 {READOUT} 均值={scores.mean():.3f}  (BACE 训练真值均值={y_train.mean():.3f})")
    plt.figure(figsize=(5, 3))
    plt.hist(y_train, bins=20, alpha=0.45, density=True, label="BACE train true pIC50")
    plt.hist(scores, bins=20, alpha=0.45, density=True, label="gen scored pIC50")
    plt.legend()
    plt.title(f"{tag}: oracle is the scorer, not experiment")
    plt.tight_layout()
    plt.show()


generated_smiles = []
for _ in range(NEW_SAMPLE):
    token_ids = gpt.generate(temperature=0.9)
    generated_smiles.append(tok_gen.decode(token_ids))    # decode：查阅词表 将ids → token，跳过特殊token → 最后连成一条smiles
evaluate_generated(generated_smiles, tag="uncond")

## 5. 条件生成：`<act_high>` / `<act_low>`（在 BACE 上微调）

ZINC **没有** pIC50，不能在大库上直接学高低活性控制符。做法是：无条件 GPT 已在大库上学写字；再在 BACE 训练 SMILES 前加上标签、微调。  

标签丢掉中间带：训练 pIC50 **最低约 30%** → `<act_low>`，**最高约 30%** → `<act_high>`，中间约 40% 不参与微调，避免活性接近却贴相反标签。可用条数变少，控制仍可能偏弱。


In [ ]:
q_low_pic50 = float(np.quantile(y_train, 0.30))
q_high_pic50 = float(np.quantile(y_train, 0.70))
print(
    f"BACE train pIC50  low≤{q_low_pic50:.3f} (bottom 30%)  "
    f"high≥{q_high_pic50:.3f} (top 30%)  drop middle ~40%"
)


class CondGenDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, q_low, q_high):
        smiles_list = list(smiles_list)
        labels = np.asarray(labels, dtype=np.float32)
        keep = (labels <= q_low) | (labels >= q_high)   # 丢掉中间带
        self.smiles_list = [smi for smi, flag in zip(smiles_list, keep) if flag]
        self.labels = labels[keep]
        self.tokenizer = tokenizer
        self.q_high = q_high
        print(f"CondGen keep {int(keep.sum())}/{len(labels)}  low={int((labels <= q_low).sum())}  high={int((labels >= q_high).sum())}")

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):
        tag = HIGH_ID if self.labels[idx] >= self.q_high else LOW_ID
        ids = self.tokenizer.encode(self.smiles_list[idx], add_special=True, pad=False)   # encode 时 add_special=True加的是首尾ids
        ids = [tag] + ids  # act 标签作为开头
        input_ids = torch.tensor(ids[:-1], dtype=torch.long)  # 学习高/低标签
        target_ids = torch.tensor(ids[1:], dtype=torch.long)
        return input_ids, target_ids


# 复制无条件权重微调；tok_gen已通过extra_special=true在词表中添加过 act 标签，无需重建 Embedding
gpt_cond = MolGPT(len(tok_gen.vocab), max_len=MAX_LEN, pad_idx=PAD_ID_G).to(DEVICE)
# 把已经在 ZINC 上训好的无条件 gpt 的全部参数拷进 gpt_cond
gpt_cond.load_state_dict(gpt.state_dict())
opt_gpt_cond = torch.optim.Adam(gpt_cond.parameters(), lr=5e-4)
train_loader_cond = DataLoader(
    CondGenDataset(smiles_bace_train, y_train, tok_gen, q_low_pic50, q_high_pic50),
    batch_size=32, shuffle=True, collate_fn=collate_shift,
)
for epoch in range(1, EPOCHS_COND + 1):
    gpt_cond.train()
    total = 0.0
    for input_ids, target_ids in train_loader_cond:
        input_ids, target_ids = input_ids.to(DEVICE), target_ids.to(DEVICE)
        opt_gpt_cond.zero_grad()
        logits = gpt_cond(input_ids)
        loss = ce(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(gpt_cond.parameters(), 1.0)
        opt_gpt_cond.step()
        total += loss.item()
    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS_COND:
        print(f"[CondGPT] epoch {epoch:3d}  CE={total / max(len(train_loader_cond), 1):.4f}")


def gen_from_tag(tag_id, n=100, temperature=0.85):
    smiles = []
    for _ in range(n):
        token_ids = gpt_cond.generate(temperature=temperature, prefix_ids=[tag_id, SOS_ID_G])
        smiles.append(tok_gen.decode(token_ids))
    return score_pic50(smiles)


scores_high, smiles_high = gen_from_tag(HIGH_ID)
scores_low, smiles_low = gen_from_tag(LOW_ID)
print(f"<act_high> 有效 {len(smiles_high)}  预测 pIC50 均值={scores_high.mean() if len(scores_high) else float('nan'):.3f}")
print(f"<act_low>  有效 {len(smiles_low)}  预测 pIC50 均值={scores_low.mean() if len(scores_low) else float('nan'):.3f}")
print("两均值接近：小数据上条件符经常学不稳，不代表打分器坏了。")

## 6. 分子优化（这才是「改已有分子」）

**生成**：无种子，从 `<sos>` 写新串。  
**优化**：有种子。构造训练对：同一训练集里找 **Morgan 相似、且 pIC50 明显更高** 的分子。Encoder 读弱的，Decoder 写更强的。

这不是对接，也不是保证改完真能抑制 BACE；是在学「打分器眼里的变强方向」，并很容易复述训练对。


In [ ]:
def smiles_to_fp_arr(smi: str):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    fp = morgan_gen.GetFingerprint(mol)   # fp 是 RDKit 的 ExplicitBitVect（显式比特向量），不是 NumPy 数组，也不是 Python 的 list
    arr = np.zeros((2048,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)   # 把分子指纹转成numpy数组
    return arr


fps = [smiles_to_fp_arr(smi) for smi in smiles_bace_train]

# 用 NumPy 算两条 Morgan 指纹的 Tanimoto 相似度（0～1，越大越像）
def tanimoto_np(fp_a, fp_b):
    if fp_a is None or fp_b is None:
        return 0.0
    inter = np.minimum(fp_a, fp_b).sum()  # np.minimum(a, b)：两个数组（或数组与标量）逐元素取小
    union = np.maximum(fp_a, fp_b).sum()
    return float(inter / union) if union > 0 else 0.0   # inter / union: 两边都有的结构个数 / 至少一边有的环境个数（交集 / 并集）


# pIC50 每差 1 ≈ 效力差约 10 倍；+0.5 作为「明显更强」的演示阈值
DELTA = 0.5      # 避免「结构像但活性差不多」，那种对学不到「往强了改」
MIN_SIMILAR = 0.40   # Tanimoto必须先超过 0.40，才有资格当「近邻」。避免「活性高但骨架完全不同」，那种对学的是换分子，不是改分子
pairs = []

# 训练集中每个分子找出活性大于0.5，相似度最高的分子对
for idx_src in range(len(smiles_bace_train)):          # 每一个训练分子当 种子 idx_src
    best_idx, best_similar = None, MIN_SIMILAR
    for idx_cand in range(len(smiles_bace_train)):    # 扫遍训练集里 别的 分子当 候选 idx_cand
        if idx_src == idx_cand:                       # 跳过自己
            continue
        if y_train[idx_cand] < y_train[idx_src] + DELTA:   # 活性不够高，跳过
            continue
        similar = tanimoto_np(fps[idx_src], fps[idx_cand])
        if similar > best_similar:                       # 相似度大于当前分子则替换相似度和候选分子索引
            best_similar, best_idx = similar, idx_cand
    if best_idx is not None:
        pairs.append((smiles_bace_train[idx_src], smiles_bace_train[best_idx]))   # 列表内以二元组形式储存

print(f"优化训练对: {len(pairs)}  (目标 pIC50 至少 +{DELTA}，且最近邻 Tanimoto>{MIN_SIMILAR})")
if len(pairs) < 8:
    print("对数太少会跳过 Seq2Seq。")


class PairDataset(Dataset):
    def __init__(self, pairs, tokenizer):
        self.pairs = pairs
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_smi, tgt_smi = self.pairs[idx]
        src_ids = self.tokenizer.encode(src_smi, add_special=True, pad=True)
        tgt_ids = self.tokenizer.encode(tgt_smi, add_special=True, pad=True)
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)


class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, pad_idx=0, max_len=128):
        super().__init__()
        self.pad_idx = pad_idx
        self.max_len = max_len
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = nn.Embedding(max_len, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=256,
            batch_first=True, dropout=0.1,
        )
        self.head = nn.Linear(d_model, vocab_size)

    # embedding + 位置编码，供Seq2Seq 的 encoder / decoder 共用
    def _append_pos(self, input_ids):
        batch_size, seq_len = input_ids.shape
        # 构造位置索引，-1 在 expand扩展广播 / reshape 里都表示：这一维保持原大小，不要我指定数字。
        pos = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        return self.embedding(input_ids) * math.sqrt(self.d_model) + self.pos_encoder(pos)

    def forward(self, src_ids, decoder_input_ids):
        src_pad_mask = src_ids.eq(self.pad_idx)
        tgt_pad_mask = decoder_input_ids.eq(self.pad_idx)
        seq_len = decoder_input_ids.size(1)
        # decoder 必须因果：位置 t 只能看 ≤ t，不能偷看未来
        # generate_square_subsequent_mask得到 [seq_len, seq_len] 上三角 mask（PyTorch Transformer 里 True = 屏蔽）
        causal = self.transformer.generate_square_subsequent_mask(seq_len).to(src_ids.device)
        # nn.Transformer 一次跑 encoder + decoder

        hidden = self.transformer(
            self._append_pos(src_ids), 
            self._append_pos(decoder_input_ids),
            tgt_mask=causal, 
            src_key_padding_mask=src_pad_mask, 
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )
        return self.head(hidden)

    '''
    完整transformer核心就三句话：
    Encoder 只负责「读懂源序列」，训/推都喂完整源。
    Decoder 负责「在源的条件下写目标」；训练时喂的是标准答案的前缀（teacher forcing），不是模型自己刚写出来的字。
    真正「生成新序列」发生在推理：decoder 从 <sos> 一步步把预测接到输入上；训练时并不逐步采样。
    '''
    
    @torch.no_grad()
    # 推理时只输入src_ids，tgt_ids都为 sos，不再强分子前缀获取信息
    def greedy(self, src_ids, max_new_tokens=128):
        self.eval()
        src_ids = torch.tensor([src_ids], dtype=torch.long, device=DEVICE)
        tgt_ids = torch.tensor([[SOS_ID_G]], dtype=torch.long, device=DEVICE)
        for _ in range(max_new_tokens):
            logits = self(src_ids, tgt_ids)[:, -1, :]
            next_id = int(logits.argmax(-1).item())
            tgt_ids = torch.cat([tgt_ids, torch.tensor([[next_id]], device=DEVICE)], dim=1)
            if next_id == EOS_ID_G:
                break
        return tgt_ids[0].tolist()


if len(pairs) >= 8:
    train_loader_s2s = DataLoader(PairDataset(pairs, tok_gen), batch_size=16, shuffle=True)
    s2s = Seq2Seq(len(tok_gen.vocab), pad_idx=PAD_ID_G, max_len=MAX_LEN).to(DEVICE)
    opt_s2s = torch.optim.Adam(s2s.parameters(), lr=1e-3)
    for epoch in range(1, EPOCHS_SEQ2SEQ + 1):
        s2s.train()
        total = 0.0
        for src_ids, tgt_ids in train_loader_s2s:
            src_ids, tgt_ids = src_ids.to(DEVICE), tgt_ids.to(DEVICE)
            opt_s2s.zero_grad()
            logits = s2s(src_ids, tgt_ids[:, :-1])
            loss = ce(logits.reshape(-1, logits.size(-1)), tgt_ids[:, 1:].reshape(-1))   # tgt_ids[:, 1:]实现错位
            loss.backward()
            nn.utils.clip_grad_norm_(s2s.parameters(), 1.0)
            opt_s2s.step()
            total += loss.item()
        if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS_SEQ2SEQ:
            print(f"[Seq2Seq] epoch {epoch:3d}  CE={total / max(len(train_loader_s2s), 1):.4f}")

    print("\n训练对上的优化演示（可能接近复述目标；看的是打分器 pIC50）：")
    for src_smi, tgt_smi in pairs[:4]:
        src_ids = tok_gen.encode(src_smi, add_special=True, pad=True)
        pred_ids = s2s.greedy(src_ids)
        pred_smi = tok_gen.decode(pred_ids)
        scores_src, _ = score_pic50([src_smi])
        scores_pred, _ = score_pic50([pred_smi])
        scores_tgt, _ = score_pic50([tgt_smi])
        score_pred = float(scores_pred[0]) if len(scores_pred) else float("nan")
        print(f"  in     {src_smi[:55]}")
        print(f"  pred   {pred_smi[:55]}  scorer={score_pred:.2f}")
        print(f"  target {tgt_smi[:55]}")
        print(
            f"  {READOUT}  src={float(scores_src[0]):.2f}  pred={score_pred:.2f}  tgt={float(scores_tgt[0]):.2f}\n"
        )
else:
    print("跳过 Seq2Seq。")

## 7. 小结

> 主靶 BACE-1，主读出 pIC50。用 scaffold 划分训练双向 Transformer 打分器；在无标签 SMILES 上预训练因果 GPT 做生成；用「相似且更高 pIC50」的分子对做 Seq2Seq 优化。报告有效率、新颖性，以及 **打分器** 预测的 pIC50。

**不要写**：生成了 BACE 药物、优化一定更强。应写清：语料仍有限、优化可能复述训练对、oracle 有误差会传到排序。

与 `07`：回归头同一类，标签从 logS 换成 pIC50，划分从随机改成骨架。  
二分类（`Class`）本子没训：若要做 hit/non-hit，换 BCE 头即可，仍是这一靶。
